# Category 2 Predictive Analysis — Interaction Experience

## Analytical statement
**Customers who enjoy service interactions are more likely to return.**

## Category 2 focus
This notebook analyzes the **Interaction Experience** dataset to predict whether a customer is retained or not retained based on interaction-quality variables such as:

- customer sentiment
- customer tone
- outcome quality
- agent politeness
- agent empathy
- agent professionalism
- communication clarity
- helpfulness
- apology, patience, active listening, and de-escalation indicators

The target variable is:

`retention_label`

where:

- `1` = retained customer
- `0` = not retained customer

## Notebook workflow

1. Load the dataset  
2. Understand missing values, duplicates, and target distribution  
3. Clean the dataset and convert the target into binary format  
4. Perform EDA using charts and correlation heatmaps  
5. Split the data into training and testing sets  
6. Build preprocessing and machine learning pipelines  
7. Compare baseline models, normal models, class-weighted models, and SMOTE models  
8. Tune model hyperparameters using cross-validation  
9. Evaluate the final best model  
10. Analyze feature importance  
11. Save model comparison and importance outputs

# 1. Environment setup
Install and import the required libraries.

In [ ]:
!pip -q install imbalanced-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

# 2. Load dataset

Upload `dimension_02_dataset.csv` when Colab asks for the file.

If the file already exists in `/content/`, the notebook will load it directly.

In [ ]:
DATA_PATH = Path("/content/dimension_02_dataset.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception:
        raise FileNotFoundError(
            "Dataset not found. Upload dimension_02_dataset.csv or update DATA_PATH."
        )

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df_raw.shape)

display(df_raw.head())

# 3. Initial dataset understanding

This section checks:

- column names
- data types
- missing values
- duplicate rows
- original target distribution

This is important before modelling because we must understand the quality and structure of the dataset.

In [ ]:
print("Column names:")
print(df_raw.columns.tolist())

print("\nDataset information:")
display(df_raw.info())

print("\nMissing values:")
display(df_raw.isna().sum())

print("\nDuplicate rows:")
print(df_raw.duplicated().sum())

print("\nOriginal target distribution:")
display(df_raw["retention_label"].value_counts(dropna=False))

print("\nOriginal target distribution percentage:")
display((df_raw["retention_label"].value_counts(normalize=True, dropna=False) * 100).round(2))

## Initial output interpretation

From the current dataset output:

- The dataset has **1000 rows and 16 columns**.
- There are **no missing values** in the dataset.
- The target variable originally has three labels:
  - `retained`
  - `not_retained`
  - `uncertain`
- `uncertain` rows are removed because this notebook performs **binary classification**.
- The dataset also contains duplicate rows. For the main model, duplicates are kept to match the original analysis, but a duplicate-removal sensitivity test can be done later if needed.

# 4. Configuration

Define target column, feature groups, and category orders.

The features are separated into:

- **Binary columns**: already represented as 0/1 indicators
- **Ordinal columns**: categories with meaningful order
- **Nominal columns**: categories with no natural order

In [ ]:
TARGET = "retention_label"

REQUIRED_COLUMNS = [
    "issue_category",
    "customer_sentiment",
    "issue_complexity",
    "agent_experience_level",
    "agent_politeness",
    "agent_empathy",
    "agent_professionalism",
    "communication_clarity",
    "apology_present",
    "patience_present",
    "active_listening_present",
    "deescalation_present",
    "helpfulness",
    "customer_tone_observed",
    "outcome_quality",
    "retention_label",
]

BINARY_FEATURES = [
    "apology_present",
    "patience_present",
    "active_listening_present",
    "deescalation_present",
]

ORDINAL_FEATURES = [
    "issue_complexity",
    "agent_experience_level",
    "customer_sentiment",
    "customer_tone_observed",
    "outcome_quality",
]

ORDINAL_CATEGORIES = [
    ["less", "medium", "high"],
    ["inexperienced", "junior", "experienced"],
    ["frustrated", "negative", "neutral", "positive"],
    ["negative", "neutral", "positive"],
    ["unknown", "unresolved", "partially_resolved", "resolved"],
]

NOMINAL_FEATURES = [
    "issue_category",
    "agent_politeness",
    "agent_empathy",
    "agent_professionalism",
    "communication_clarity",
    "helpfulness",
]

TEXT_COLUMNS_TO_CLEAN = [
    "issue_category",
    "customer_sentiment",
    "issue_complexity",
    "agent_experience_level",
    "agent_politeness",
    "agent_empathy",
    "agent_professionalism",
    "communication_clarity",
    "helpfulness",
    "customer_tone_observed",
    "outcome_quality",
    "retention_label",
]

TARGET_MAPPING = {
    "not_retained": 0,
    "retained": 1,
}

DROP_DUPLICATES = False

# 5. Data cleaning and target preparation

Main cleaning steps:

1. Strip spaces and convert text values to lowercase.
2. Remove rows labelled `uncertain`.
3. Map:
   - `not_retained` → `0`
   - `retained` → `1`
4. Convert binary indicator columns to integer format.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()

missing_required = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

for col in TEXT_COLUMNS_TO_CLEAN:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Remove uncertain labels to create a binary classification task
df = df[df[TARGET] != "uncertain"].copy()

# Convert target to 0/1
df[TARGET] = df[TARGET].map(TARGET_MAPPING)

if df[TARGET].isna().any():
    raise ValueError("Unexpected values found in retention_label after mapping.")

# Convert binary indicator columns to integers
for col in BINARY_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="raise").astype(int)

if DROP_DUPLICATES:
    before_rows = len(df)
    df = df.drop_duplicates().copy()
    print(f"Duplicate rows removed: {before_rows - len(df)}")

print("Cleaned dataset shape:", df.shape)

print("\nTarget distribution after binary conversion:")
display(df[TARGET].value_counts())

print("\nTarget percentage:")
display((df[TARGET].value_counts(normalize=True) * 100).round(2))

display(df.head())

## Cleaned dataset output interpretation

After removing `uncertain` rows, the binary dataset contains:

- **702 retained customers**
- **112 not-retained customers**
- **814 total records**

This means the dataset is **imbalanced** because retained customers are the majority class.

# 6. Exploratory Data Analysis — Target distribution

The target distribution chart shows whether the data is balanced or imbalanced.

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Not Retained", "Retained"], target_counts.values)
plt.title("Target Class Distribution - Interaction Experience Dataset")
plt.xlabel("Retention Label")
plt.ylabel("Number of Customers")
plt.show()

## Target distribution conclusion

The retained class is much larger than the not-retained class. Therefore, model evaluation should not depend only on **accuracy**.  

For this reason, this notebook also uses:

- balanced accuracy
- macro F1-score
- confusion matrix
- ROC-AUC

# 7. EDA — Categorical feature vs retention

These plots show how each important interaction-experience feature is distributed across retained and not-retained customers.

In [ ]:
def plot_categorical_by_target(column_name):
    cross_tab = pd.crosstab(df[column_name], df[TARGET], normalize="index") * 100

    cross_tab.plot(kind="bar", figsize=(8, 4))
    plt.title(f"{column_name} vs Retention Label")
    plt.xlabel(column_name)
    plt.ylabel("Percentage")
    plt.legend(["Not Retained", "Retained"])
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    display(cross_tab.round(2))

In [ ]:
important_eda_features = [
    "customer_sentiment",
    "customer_tone_observed",
    "outcome_quality",
    "issue_complexity",
]

for feature in important_eda_features:
    plot_categorical_by_target(feature)

## Categorical EDA conclusion

From the current output:

- **Outcome quality** is the clearest separating factor. `resolved` cases are mostly retained, while `unresolved`, `unknown`, and `partially_resolved` cases are mostly not retained.
- **Customer tone** and **customer sentiment** also show useful patterns. More positive/neutral interaction signals are linked with retention.
- **Issue complexity** does not show a very strong separation compared with outcome quality.

# 8. EDA — Binary service behaviour indicators

These features check whether service behaviours such as apology, patience, active listening, and de-escalation are related to retention.

In [ ]:
for feature in BINARY_FEATURES:
    plot_categorical_by_target(feature)

## Binary indicator conclusion

The binary behaviour indicators have some relationship with retention, but they are not as strong as `outcome_quality`.

Some binary features may appear counterintuitive because an apology or de-escalation is usually present when the customer already has a problem. So these variables must be interpreted together with issue severity and outcome quality.

# 9. EDA — Correlation heatmap

For correlation analysis, ordinal variables are temporarily mapped into numeric values and nominal variables are one-hot encoded.

The full heatmap gives an overall view.  
The target-focused heatmap is easier to use in the report because it shows which variables have stronger positive or negative relationships with retention.

In [ ]:
eda_df = df.copy()

ordinal_mappings = {
    "issue_complexity": {
        "less": 0,
        "medium": 1,
        "high": 2,
    },
    "agent_experience_level": {
        "inexperienced": 0,
        "junior": 1,
        "experienced": 2,
    },
    "customer_sentiment": {
        "frustrated": 0,
        "negative": 1,
        "neutral": 2,
        "positive": 3,
    },
    "customer_tone_observed": {
        "negative": 0,
        "neutral": 1,
        "positive": 2,
    },
    "outcome_quality": {
        "unknown": 0,
        "unresolved": 1,
        "partially_resolved": 2,
        "resolved": 3,
    },
}

for col, mapping in ordinal_mappings.items():
    eda_df[col] = eda_df[col].map(mapping)

eda_encoded = pd.get_dummies(
    eda_df,
    columns=NOMINAL_FEATURES,
    dtype=int
)

numeric_corr_df = eda_encoded.select_dtypes(include=["number"])
corr_matrix = numeric_corr_df.corr(method="pearson")

plt.figure(figsize=(18, 14))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    annot=False,
    linewidths=0.5
)
plt.title("Correlation Heatmap - Interaction Experience Dataset")
plt.show()

In [ ]:
target_corr = corr_matrix[[TARGET]].sort_values(
    by=TARGET,
    ascending=False
)

plt.figure(figsize=(8, 12))
sns.heatmap(
    target_corr,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5
)
plt.title("Feature Correlation with Retention Label")
plt.show()

display(target_corr.round(3))

## Correlation heatmap conclusion

From the current output, the strongest positive relationship with retention is:

- `outcome_quality`

Other useful positive signals include:

- high helpfulness
- high communication clarity
- high agent politeness
- positive customer tone
- positive customer sentiment

Some medium/low/unknown service-quality categories show negative relationships with retention. This supports the idea that better interaction quality is related to customer return behaviour.

# 10. Define features and target

`X` contains the input features.  
`y` contains the target variable.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
display(y.value_counts())

print("\nTarget percentage:")
display((y.value_counts(normalize=True) * 100).round(2))

# 11. Check class imbalance

If the minority class percentage is low, SMOTE models are included for comparison.

In [ ]:
minority_percentage = y.value_counts(normalize=True).min()

print("Minority class percentage:", round(minority_percentage * 100, 2), "%")

USE_SMOTE = minority_percentage < 0.35

print("Use SMOTE:", USE_SMOTE)

## Imbalance conclusion

The minority class is around **13.76%**, so this is an imbalanced classification problem.  

Therefore, the model comparison includes:

- normal models
- class-weighted models
- SMOTE-based models

# 12. Train-test split

The dataset is split into:

- 80% training data
- 20% testing data

`stratify=y` keeps the class ratio similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTrain target distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest target distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))

# 13. Preprocessing pipeline

Different preprocessing is applied to different feature types:

## Binary features
- Most frequent imputation

## Ordinal features
- Most frequent imputation
- Ordinal encoding
- Standard scaling

## Nominal features
- Most frequent imputation
- One-hot encoding

In [ ]:
set_config(transform_output="pandas")

binary_transformer = SkPipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

ordinal_transformer = SkPipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ORDINAL_CATEGORIES,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        ),
        ("scaler", StandardScaler())
    ]
)

nominal_transformer = SkPipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("binary", binary_transformer, BINARY_FEATURES),
        ("ordinal", ordinal_transformer, ORDINAL_FEATURES),
        ("nominal", nominal_transformer, NOMINAL_FEATURES)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# 14. Model pipeline function

This function creates a complete workflow:

raw data → preprocessing → optional SMOTE → model training

SMOTE is placed inside the pipeline so it is applied only during training, not directly to the test data.

In [ ]:
def make_pipeline(model, use_smote=False):
    steps = [
        ("preprocess", preprocessor)
    ]

    if use_smote:
        steps.append(
            (
                "smote",
                SMOTE(
                    random_state=RANDOM_STATE,
                    k_neighbors=5
                )
            )
        )

    steps.append(("model", model))

    return ImbPipeline(steps=steps)

# 15. Model evaluation function

This function trains a model and calculates:

- accuracy
- balanced accuracy
- precision
- recall
- F1-score
- macro F1-score
- ROC-AUC
- classification report
- confusion matrix
- ROC curve

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test, show_report=True):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    results = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_retained_1": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_retained_1": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_retained_1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    }

    if show_report:
        print("=" * 80)
        print(name)
        print("=" * 80)

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, zero_division=0))

        cm = confusion_matrix(y_test, y_pred)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Not Retained", "Retained"]
        )

        disp.plot()
        plt.title(f"Confusion Matrix - {name}")
        plt.show()

        if y_proba is not None:
            RocCurveDisplay.from_predictions(y_test, y_proba)
            plt.title(f"ROC Curve - {name}")
            plt.show()

    return results, model

# 16. Build baseline and machine learning models

Baseline models are included to check whether the machine learning models are better than simple guessing.

Models compared:

1. Most Frequent Class baseline  
2. Stratified Random baseline  
3. Logistic Regression  
4. Class-weighted Logistic Regression  
5. Random Forest  
6. Class-weighted Random Forest  
7. Logistic Regression + SMOTE  
8. Random Forest + SMOTE

In [ ]:
models = {
    "Baseline: Most Frequent Class": make_pipeline(
        DummyClassifier(
            strategy="most_frequent",
            random_state=RANDOM_STATE
        )
    ),

    "Baseline: Stratified Random": make_pipeline(
        DummyClassifier(
            strategy="stratified",
            random_state=RANDOM_STATE
        )
    ),

    "Logistic Regression": make_pipeline(
        LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    ),

    "Logistic Regression - Class Weighted": make_pipeline(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),

    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),

    "Random Forest - Class Weighted": make_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
}

if USE_SMOTE:
    models["Logistic Regression + SMOTE"] = make_pipeline(
        LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        ),
        use_smote=True
    )

    models["Random Forest + SMOTE"] = make_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        use_smote=True
    )

# 17. Train and compare all models

The models are sorted mainly by:

1. macro F1-score  
2. balanced accuracy  
3. ROC-AUC  

These metrics are better than accuracy alone because the target classes are imbalanced.

In [ ]:
all_results = []
trained_models = {}

for name, model in models.items():
    result, fitted_model = evaluate_model(
        name,
        model,
        X_train,
        X_test,
        y_train,
        y_test,
        show_report=False
    )

    all_results.append(result)
    trained_models[name] = fitted_model

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by=["f1_macro", "balanced_accuracy", "roc_auc"],
    ascending=False
).reset_index(drop=True)

display(results_df)

## Model comparison interpretation

From the current output:

- The most frequent baseline gives high accuracy because it predicts the majority class, but its balanced accuracy is only **0.50**.
- This proves that accuracy alone is misleading for this dataset.
- The best pre-tuning model is **Random Forest + SMOTE**.
- Random Forest models perform better than Logistic Regression overall.

# 18. Visual model comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(results_df["model"], results_df["f1_macro"])
plt.xlabel("Macro F1 Score")
plt.ylabel("Model")
plt.title("Model Comparison Based on Macro F1 Score")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(results_df["model"], results_df["balanced_accuracy"])
plt.xlabel("Balanced Accuracy")
plt.ylabel("Model")
plt.title("Model Comparison Based on Balanced Accuracy")
plt.gca().invert_yaxis()
plt.show()

# 19. Best model before tuning

This section prints the full classification report, confusion matrix, and ROC curve for the best model before hyperparameter tuning.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Best model before tuning:", best_model_name)

best_result, best_model = evaluate_model(
    best_model_name,
    best_model,
    X_train,
    X_test,
    y_train,
    y_test,
    show_report=True
)

## Best pre-tuning model interpretation

From the current output, **Random Forest + SMOTE** predicts almost all test samples correctly.

The confusion matrix result is effectively:

- All **22 not-retained** customers were correctly identified.
- **140 out of 141 retained** customers were correctly identified.
- Only **1 retained customer** was misclassified.

This is a very strong result.

# 20. Hyperparameter tuning setup

GridSearchCV is used with stratified cross-validation.

The main refit metric is:

`f1_macro`

because macro F1 gives importance to both classes, not only the majority class.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc"
}

# 21. Hyperparameter tuning candidates

Two models are tuned:

1. Logistic Regression + SMOTE  
2. Random Forest + SMOTE

In [ ]:
tuning_candidates = {}

logistic_pipeline = make_pipeline(
    LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE
    ),
    use_smote=USE_SMOTE
)

logistic_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear", "lbfgs"]
}

if USE_SMOTE:
    logistic_grid["smote__k_neighbors"] = [3, 5]

tuning_candidates["Tuned Logistic Regression"] = (
    logistic_pipeline,
    logistic_grid
)


rf_pipeline = make_pipeline(
    RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    use_smote=USE_SMOTE
)

rf_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

if USE_SMOTE:
    rf_grid["smote__k_neighbors"] = [3, 5]

tuning_candidates["Tuned Random Forest"] = (
    rf_pipeline,
    rf_grid
)

# 22. Run hyperparameter tuning

This step can take some time because Random Forest tests many parameter combinations.

In [ ]:
tuned_results = []
tuned_models = {}
best_params_summary = {}

for name, (pipeline, param_grid) in tuning_candidates.items():
    print("=" * 80)
    print("Tuning:", name)
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    print("\nBest parameters:")
    print(grid_search.best_params_)

    print("\nBest CV Macro F1:")
    print(grid_search.best_score_)

    best_params_summary[name] = {
        "best_params": grid_search.best_params_,
        "best_cv_macro_f1": grid_search.best_score_
    }

    best_tuned_model = grid_search.best_estimator_

    result, fitted_model = evaluate_model(
        name,
        best_tuned_model,
        X_train,
        X_test,
        y_train,
        y_test,
        show_report=False
    )

    tuned_results.append(result)
    tuned_models[name] = fitted_model

## Hyperparameter tuning interpretation

From the current output:

- Tuned Logistic Regression best CV Macro F1 ≈ **0.9677**
- Tuned Random Forest best CV Macro F1 ≈ **0.9635**

Even though Logistic Regression had slightly higher cross-validation macro F1, the tuned Random Forest performed best on the final test comparison.

# 23. Final comparison after tuning

In [ ]:
tuned_results_df = pd.DataFrame(tuned_results)

final_comparison_df = pd.concat(
    [results_df, tuned_results_df],
    ignore_index=True
)

final_comparison_df = final_comparison_df.sort_values(
    by=["f1_macro", "balanced_accuracy", "roc_auc"],
    ascending=False
).reset_index(drop=True)

display(final_comparison_df)

## Final comparison conclusion

From the current output, the best final model is:

**Tuned Random Forest**

It achieved approximately:

- Accuracy: **0.9939**
- Balanced Accuracy: **0.9965**
- Macro F1-score: **0.9871**
- ROC-AUC: **0.9987**

This means the model can predict customer retention very strongly using interaction-experience variables.

# 24. Final best model evaluation

This gives the final classification report, confusion matrix, and ROC curve.

In [ ]:
final_best_model_name = final_comparison_df.iloc[0]["model"]

if final_best_model_name in tuned_models:
    final_best_model = tuned_models[final_best_model_name]
else:
    final_best_model = trained_models[final_best_model_name]

print("Final best model:", final_best_model_name)

final_result, final_best_model = evaluate_model(
    final_best_model_name,
    final_best_model,
    X_train,
    X_test,
    y_train,
    y_test,
    show_report=True
)

## Final model interpretation

The final tuned Random Forest performed extremely well.

From the current confusion matrix:

- **22 / 22 not-retained customers** were correctly predicted.
- **140 / 141 retained customers** were correctly predicted.
- Only **1 retained customer** was incorrectly classified as not retained.

This supports the analytical statement because interaction-experience features are highly useful for predicting customer return/retention.

# 25. Permutation feature importance

Permutation importance checks how much model performance drops when each original feature is shuffled.

If shuffling a feature reduces performance a lot, that feature is important.

In [ ]:
perm_result = permutation_importance(
    final_best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

perm_importance_df = perm_importance_df.sort_values(
    by="importance_mean",
    ascending=False
).reset_index(drop=True)

display(perm_importance_df)

In [ ]:
top_perm = perm_importance_df.head(10).sort_values("importance_mean")

plt.figure(figsize=(8, 5))
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.xlabel("Permutation Importance Mean")
plt.ylabel("Feature")
plt.title("Top Original Features by Permutation Importance")
plt.show()

## Permutation importance conclusion

From the current output, the most important feature is:

**outcome_quality**

Other useful predictors are:

- issue category
- helpfulness
- agent politeness
- customer sentiment
- issue complexity
- customer tone observed

This means the final result of the interaction — especially whether the issue was resolved — is the strongest predictor of customer retention.

# 26. Model-based feature importance

For Random Forest, model-based feature importance is calculated using the transformed features after preprocessing.

In [ ]:
def get_transformed_feature_names(fitted_pipeline):
    return fitted_pipeline.named_steps["preprocess"].get_feature_names_out()


def plot_model_based_importance(fitted_pipeline, top_n=20):
    model = fitted_pipeline.named_steps["model"]
    feature_names = get_transformed_feature_names(fitted_pipeline)

    if hasattr(model, "feature_importances_"):
        importance_values = model.feature_importances_

        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": importance_values
        }).sort_values("importance", ascending=False).reset_index(drop=True)

        top_df = importance_df.head(top_n).sort_values("importance")

        plt.figure(figsize=(10, 6))
        plt.barh(top_df["feature"], top_df["importance"])
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.title("Model-Based Feature Importance")
        plt.show()

        return importance_df

    elif hasattr(model, "coef_"):
        coef_values = model.coef_[0]

        importance_df = pd.DataFrame({
            "feature": feature_names,
            "coefficient": coef_values,
            "absolute_coefficient": np.abs(coef_values)
        }).sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)

        top_df = importance_df.head(top_n).sort_values("absolute_coefficient")

        plt.figure(figsize=(10, 6))
        plt.barh(top_df["feature"], top_df["absolute_coefficient"])
        plt.xlabel("Absolute Coefficient")
        plt.ylabel("Feature")
        plt.title("Logistic Regression Feature Importance")
        plt.show()

        return importance_df

    else:
        print("This model does not provide direct feature importance or coefficients.")
        return None

In [ ]:
model_importance_df = plot_model_based_importance(final_best_model, top_n=20)

if model_importance_df is not None:
    display(model_importance_df.head(20))

# 27. Save final outputs

The model comparison and feature importance tables are saved as CSV files.

In [ ]:
final_comparison_df.to_csv("category_02_interaction_experience_model_comparison.csv", index=False)
perm_importance_df.to_csv("category_02_interaction_experience_permutation_importance.csv", index=False)

if model_importance_df is not None:
    model_importance_df.to_csv("category_02_interaction_experience_model_importance.csv", index=False)

print("Saved output files:")
print("category_02_interaction_experience_model_comparison.csv")
print("category_02_interaction_experience_permutation_importance.csv")

if model_importance_df is not None:
    print("category_02_interaction_experience_model_importance.csv")

In [ ]:
try:
    from google.colab import files

    files.download("category_02_interaction_experience_model_comparison.csv")
    files.download("category_02_interaction_experience_permutation_importance.csv")

    if model_importance_df is not None:
        files.download("category_02_interaction_experience_model_importance.csv")
except Exception:
    print("Download skipped. Files were saved in the current working directory.")

# 28. Final report-ready conclusion

For Category 2, **Interaction Experience**, predictive analytics was conducted to determine whether service interaction quality can predict customer retention. The target variable was `retention_label`, where retained customers were encoded as `1` and not-retained customers were encoded as `0`. Rows labelled as `uncertain` were removed to create a clear binary classification task.

The dataset was imbalanced because retained customers made up the majority of the records. Therefore, the analysis used balanced accuracy and macro F1-score in addition to normal accuracy. Baseline models were also tested to make sure the machine learning models performed better than simple guessing.

The final best model was **Tuned Random Forest**, which achieved very strong performance on the test set. The model correctly identified nearly all retained and not-retained customers. Feature importance showed that `outcome_quality` was the strongest predictor, followed by features such as issue category, helpfulness, agent politeness, customer sentiment, issue complexity, and customer tone.

Therefore, the predictive analytics results for Category 2 support the analytical statement that **customers who enjoy service interactions are more likely to return**. In this dataset, better outcome quality, helpfulness, clearer communication, and more positive customer experience indicators were strongly linked with customer retention.

# 29. Short viva answer

For Category 2, we predicted customer retention using interaction-experience variables. We removed uncertain labels and converted the target into binary form. Since the data was imbalanced, we compared baseline models, normal models, class-weighted models, and SMOTE models. We used balanced accuracy and macro F1-score because accuracy alone can be misleading in imbalanced data. The best final model was Tuned Random Forest, which achieved very strong performance. Feature importance showed that outcome quality was the most important predictor. This supports the statement that customers with better service interactions are more likely to return.